# A2 — Pierce 1890 Knowledge-Base Demo
**Team G07 · doc-agent · Assignment 2**

This notebook provides verified, fully-grounded A2 evidence:
1. **Part 1** — OCR quality metrics (CER, WER, Word-F1) evaluated across 24 held-out pages vs. `grading_kit/labels.jsonl`
2. **Part 2** — Index overview (3,830 chunks, 1024 embedding dimensions, `faiss:flat_ip`, 1,034 pages / 409,102 words)
3. **Part 3** — Live vector retrieval demo using winning `Qwen3-Embedding-0.6B`:
   - Successful retrieval examples (e.g. `q_test_02` on perspiration functions `p0078`)
   - Worst-failure retrieval case (`q_multi_test_02` hydrotherapy overview displaced by sub-modality shower bath `p0373`)
   - Out-of-corpus abstention case (`q_neg_06` modern statins mechanism with confidence below $\tau = 0.55$)

> Executed directly against the production FAISS index at `data/processed/index`.

In [1]:
import sys, json, re, html
from collections import Counter
from pathlib import Path

import yaml
import numpy as np

# Auto-detect repository root whether kernel started in . or ./notebooks
cwd = Path.cwd().resolve()
REPO = cwd.parent if not (cwd / 'configs' / 'config.yaml').is_file() and (cwd.parent / 'configs' / 'config.yaml').is_file() else cwd
sys.path.insert(0, str(REPO / 'src'))

cfg = yaml.safe_load((REPO / 'configs' / 'config.yaml').read_text())
INDEX_DIR = (REPO / Path(cfg['index']['path'])).resolve()
cfg['index']['path'] = str(INDEX_DIR)  # Ensure absolute path for store.py

LABELS_PATH = (REPO / 'grading_kit' / 'labels.jsonl').resolve()
CANONICAL_PATH = (REPO / 'data' / 'canonical-pages.jsonl').resolve()
CHANDRA_DIR = (REPO / 'chandra').resolve()
PAGES_MD = (REPO / 'chandra' / 'pages.md').resolve()
IMG_IDX_PATH = INDEX_DIR / 'image_index.json'

print('REPO root :', REPO)
print('Labels    :', LABELS_PATH.is_file(), f'({LABELS_PATH})')
print('Index dir :', INDEX_DIR.is_dir(), f'({INDEX_DIR})')
print('Config    :', f"{cfg['embed']['model']} (dim={cfg['embed']['dim']}), index={cfg['index']['type']}")


REPO root : /Users/smammahdi/CSE_stuffs/Project/DL Project/doc-agent-starter
Labels    : True (/Users/smammahdi/CSE_stuffs/Project/DL Project/doc-agent-starter/grading_kit/labels.jsonl)
Index dir : True (/Users/smammahdi/CSE_stuffs/Project/DL Project/doc-agent-starter/data/processed/index)
Config    : Qwen/Qwen3-Embedding-0.6B (dim=1024), index=faiss:flat_ip


## Part 1 — OCR Quality on Held-Out Pages

In [2]:
if not LABELS_PATH.is_file():
    raise FileNotFoundError(f'Missing {LABELS_PATH} — add hand-corrected labels.')

labels = {}
for line in LABELS_PATH.read_text('utf-8').splitlines():
    if not line.strip() or line.lstrip().startswith('#'):
        continue
    row = json.loads(line)
    labels[row['page_id']] = row['text']

print(f'Loaded {len(labels)} held-out ground truth labels: {sorted(labels.keys())[:5]} ...')


Loaded 24 held-out ground truth labels: ['p0024', 'p0025', 'p0026', 'p0027', 'p0028'] ...


In [3]:
def edit_distance(seq1, seq2):
    if len(seq1) < len(seq2):
        seq1, seq2 = seq2, seq1
    if not seq2:
        return len(seq1)
    prev = list(range(len(seq2) + 1))
    for i, c1 in enumerate(seq1):
        curr = [i + 1] * (len(seq2) + 1)
        for j, c2 in enumerate(seq2):
            cost = 0 if c1 == c2 else 1
            curr[j + 1] = min(curr[j] + 1, prev[j + 1] + 1, prev[j] + cost)
        prev = curr
    return prev[len(seq2)]

def compute_cer(hyp: str, ref: str) -> float:
    if not ref:
        return 0.0 if not hyp else 1.0
    return edit_distance(hyp, ref) / max(len(ref), 1)

def compute_wer(hyp: str, ref: str) -> float:
    hw = hyp.strip().split()
    rw = ref.strip().split()
    if not rw:
        return 0.0 if not hw else 1.0
    return edit_distance(hw, rw) / max(len(rw), 1)

def compute_word_f1(hyp: str, ref: str) -> float:
    hw = Counter(hyp.lower().split())
    rw = Counter(ref.lower().split())
    overlap = sum((hw & rw).values())
    total_p = sum(hw.values())
    total_r = sum(rw.values())
    if total_p == 0 or total_r == 0:
        return 0.0
    p = overlap / total_p
    r = overlap / total_r
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

# Load OCR predictions from canonical pages
ocr_pages = {}
if CANONICAL_PATH.is_file():
    for line in CANONICAL_PATH.read_text(encoding='utf-8').splitlines():
        if line.strip() and not line.startswith('#'):
            row = json.loads(line)
            ocr_pages[row['page_id']] = row['text']
elif PAGES_MD.is_file():
    from doc_agent.index.chunk import load_from_pages_markdown
    p_chunks, _ = load_from_pages_markdown(PAGES_MD, 'pierce-1890')
    for c in p_chunks:
        ocr_pages[c.page_ids[0]] = c.text

print(f'Total OCR pages loaded: {len(ocr_pages)}')

print('\nPage           CER     WER   Word F1')
print('-' * 38)
cers, wers, f1s = [], [], []
for pid in sorted(labels.keys()):
    ref = labels[pid]
    hyp = ocr_pages.get(pid, '')
    cer = compute_cer(hyp, ref)
    wer = compute_wer(hyp, ref)
    f1  = compute_word_f1(hyp, ref)
    cers.append(cer)
    wers.append(wer)
    f1s.append(f1)
    print(f'{pid:<10}  {cer:.4f}  {wer:.4f}    {f1:.4f}')

print('-' * 38)
print(f'Macro Mean: CER={np.mean(cers):.4f}  WER={np.mean(wers):.4f}  Word-F1={np.mean(f1s):.4f}')


Total OCR pages loaded: 1034

Page           CER     WER   Word F1
--------------------------------------
p0024       0.0266  0.0874    0.9305
p0025       0.0313  0.0732    0.9409
p0026       0.0030  0.0151    0.9864
p0027       0.1760  0.2143    0.9668
p0028       0.0091  0.0290    0.9784
p0029       0.2608  0.3368    0.9556
p0030       0.3663  0.4369    0.9323
p0031       0.2522  0.3457    0.9675
p0032       0.3842  0.5117    0.8186
p0033       0.1552  0.1761    0.9781
p0034       0.5108  0.5465    0.9596
p0035       0.0092  0.0691    0.9490
p0036       0.0110  0.0134    0.9900
p0037       0.0046  0.0259    0.9794
p0038       0.0086  0.0480    0.9633
p0039       0.1637  0.1511    0.9785
p0040       0.2450  0.3020    0.9697
p0041       0.1776  0.2778    0.8947
p0042       0.0029  0.0049    0.9963
p0043       0.1667  0.2222    0.9474
p0044       0.0040  0.0000    1.0000
p0045       0.2023  0.2773    0.9854
p0046       0.2211  0.2640    0.9592
p0047       0.0055  0.0091    0.9924
------

## Part 2 — Knowledge Base & Index Overview

In [4]:
from doc_agent.index.store import load

faiss_index, indexed_chunks, metadata = load(cfg)

# Load image index
img_idx = json.loads(IMG_IDX_PATH.read_text()) if IMG_IDX_PATH.exists() else {}
pages_with_figs = len(img_idx)
total_figs = sum(len(v) for v in img_idx.values())

print('=' * 65)
print('  STAGE 4 — FAISS KNOWLEDGE BASE PRODUCTION STATISTICS')
print('=' * 65)
print(f'  Index type         : {metadata["index_type"]}')
print(f'  Embedding model    : {cfg["embed"]["model"]}')
print(f'  Embedding dim      : {metadata["dimension"]}')
print(f'  Total chunks       : {metadata["count"]:,}')
print(f'  Chunking policy    : Fixed {cfg["index"].get("chunk_words", 128)} words / {cfg["index"].get("overlap", 16)} words overlap')
print(f'  Pages indexed      : {len(set(pid for c in indexed_chunks for pid in c.page_ids))} (1,034 total PDF pages, 1,016 non-empty)')
print(f'  Total words indexed: {sum(len(c.text.split()) for c in indexed_chunks):,}')
print(f'  Index vectors      : {faiss_index.ntotal:,}')
print('=' * 65)


  STAGE 4 — FAISS KNOWLEDGE BASE PRODUCTION STATISTICS
  Index type         : faiss:flat_ip
  Embedding model    : Qwen/Qwen3-Embedding-0.6B
  Embedding dim      : 1024
  Total chunks       : 3,830
  Chunking policy    : Fixed 128 words / 16 words overlap
  Pages indexed      : 1016 (1,034 total PDF pages, 1,016 non-empty)
  Total words indexed: 409,102
  Index vectors      : 3,830


## Part 3 — Live Vector Retrieval Demo

Retrieval uses `Qwen3-Embedding-0.6B` with normalized inner product (exact cosine similarity) and model instruction prefixing.

In [5]:
import faiss
from doc_agent.index.embed import encode_queries
from IPython.display import display, Image as IPImage, Markdown

def retrieve(query: str, k: int = 5):
    """Embed query with instruction prefix, search index, return top-k hits."""
    qvec = encode_queries([query], cfg)
    scores, positions = faiss_index.search(qvec, k)
    hits = []
    for score, pos in zip(scores[0], positions[0]):
        if pos < 0: continue
        chunk = indexed_chunks[pos]
        page_id = chunk.page_ids[0]
        hits.append({'chunk': chunk, 'score': float(score), 'page_id': page_id})
    return hits

def show_results(query: str, k: int = 3, expected_pages: list[str] | None = None):
    display(Markdown(f'### Query: *{query}*'))
    if expected_pages:
        display(Markdown(f'**Gold Target Pages**: `{expected_pages}`'))
    hits = retrieve(query, k)
    for i, h in enumerate(hits, 1):
        c = h['chunk']
        is_match = (expected_pages is not None) and (c.page_ids[0] in expected_pages)
        match_tag = ' ✅ **[CORRECT PAGE HIT]**' if is_match else ''
        display(Markdown(
            f'**Result {i}** | Chunk: `{c.id}` | Page: `{c.page_ids[0]}`{match_tag} | Cosine score: `{h["score"]:.4f}`\n\n'
            f'> {c.text[:350]}...'
        ))
    return hits

# ─── 1. Successful Real Retrieval (q_test_02 on Perspiration) ─────────────
show_results(
    'What are the two primary physiological offices of perspiration in the human body?',
    k=3,
    expected_pages=['p0078']
)


### Query: *What are the two primary physiological offices of perspiration in the human body?*


**Gold Target Pages**: `['p0078']`


**Result 1** | Chunk: `pierce-1890_p0078_c0197` | Page: `p0078` ✅ **[CORRECT PAGE HIT]** | Cosine score: `0.6929`

> 70 COMMON SENSE MEDICAL ADVISER. of it passes off as insensible transpiration, yet it often accumulates in drops of sweat, during long-continued exercise Fig. 48. A perspiratory gland, highly magnified. 1, 1. The gland. 2, 2. Excretory ducts uniting to form a tube which tortuously perforates the cuticle at 3, and opens obliquely on its surface at 4...


**Result 2** | Chunk: `pierce-1890_p0084_c0215` | Page: `p0084` | Cosine score: `0.6223`

> it is called invisible or insensible perspiration . When there is unusual muscular activity, it collects upon the skin, and is known as sensible perspiration . This secretion performs an important office in the animal economy, by maintaining the internal temperature at about 100° Fahr. Even in the Arctic regions, where the explorer has to adapt him...


**Result 3** | Chunk: `pierce-1890_p0084_c0214` | Page: `p0084` | Cosine score: `0.5966`

> Chloride of Potassium, . . . . . 0.24 Sulphate of Soda and Potassa, . . . . . 0.01 Salts of organic acids, with Soda and Potassa, . . . . . 2.02 1000.00 Traces of organic matter, mingled with a free volatile acid, are also found in the perspiration. It is the acid which imparts to this secretion its peculiar odor, and acid reaction. The process of ...


In [6]:
# ─── 2. Additional Successful Real Retrieval (q_test_04 on Tactile Sensibility) ───
show_results(
    'Which anatomical parts of the human body possess the most acute tactile sensibility according to 19th-century physiology?',
    k=3,
    expected_pages=['p0121']
)


### Query: *Which anatomical parts of the human body possess the most acute tactile sensibility according to 19th-century physiology?*


**Gold Target Pages**: `['p0121']`


**Result 1** | Chunk: `pierce-1890_p0121_c0336` | Page: `p0121` ✅ **[CORRECT PAGE HIT]** | Cosine score: `0.7124`

> tip of the tongue possesses the most acute sensibility of any portion of the body, and next in order are the tips of the fingers. The hands are the principal organs of tactile sensation. The nerves of general sensibility are distributed to every part of the cutaneous tissue. The contact of a foreign body with the back, will produce a similar tactil...


**Result 2** | Chunk: `pierce-1890_p0121_c0335` | Page: `p0121` | Cosine score: `0.6196`

> TOUCH. 113 nauseous drug may then be swallowed without experiencing any disagreeable taste. Paralysis of the facial nerve often produces a marked effect in the sensibility of the tongue. Where this influence lies has not been fully explained; probably it is indirect, being produced by some alteration in the vascularity of the parts or a diminution ...


**Result 3** | Chunk: `pierce-1890_p0120_c0331` | Page: `p0120` | Cosine score: `0.5760`

> 112 COMMON SENSE MEDICAL ADVISER. filaments which are termed respectively nerves of special and nerves of general sensation . Compared with the lower animals, especially with those belonging to the carnivorous species, the sense of smell in man is feeble. The sensation of smell is especially connected with the pleasures and necessities of animal li...


In [7]:
# ─── 3. Real Worst Failure Analysis (q_multi_test_02 on Hydrotherapy) ───────
display(Markdown('### ⚠️ Failure Analysis Case: Multi-Page Topic Dispersion'))
hits_fail = show_results(
    'What are the therapeutic purposes and physiological effects of hydrotherapy and medicinal baths according to Dr. Pierce?',
    k=5,
    expected_pages=['p0364', 'p0365', 'p0366']
)

top_hit_page = hits_fail[0]['page_id']
print(f"\nDiagnosis: Top retrieved chunk is on page {top_hit_page} (Shower Bath / Sea Bathing sub-modality).")
print("Reason: The query 'therapeutic purposes and physiological effects of hydrotherapy' caused the dense encoder")
print("to match specific violent-reaction bath descriptions on p0373 ('shock to the nervous system') rather than")
print("the general introductory definitions on p0364-p0366. In Stage 5 (A3), hybrid retrieval + BGE reranking resolves this.")


### ⚠️ Failure Analysis Case: Multi-Page Topic Dispersion


### Query: *What are the therapeutic purposes and physiological effects of hydrotherapy and medicinal baths according to Dr. Pierce?*


**Gold Target Pages**: `['p0364', 'p0365', 'p0366']`


**Result 1** | Chunk: `pierce-1890_p0373_c1173` | Page: `p0373` | Cosine score: `0.5774`

> classes unless they live near the sea-shore. The Shower Bath produces a shock to the nervous system by suddenly coming in contact with the skin. Numerous streams of cold water fall upon the neck, shoulders, and body of the patient who stands beneath the hose or reservoir. When the patient is plethoric, feeble, or nervous, or when some internal orga...


**Result 2** | Chunk: `pierce-1890_p0900_c3247` | Page: `p0900` | Cosine score: `0.5760`

> preferable. As constitutional treatment, Dr. Pierce's Golden Medical Discovery is an agent of inestimable value in this affection, although there are cases in which it may fail to effect a cure. It can not be expected that a remedy adapted to so many diseases, should be sufficiently powerful to eradicate, in every instance , this terrible scourge. ...


**Result 3** | Chunk: `pierce-1890_p0972_c3466` | Page: `p0972` | Cosine score: `0.5759`

> advise the afflicted that, in many complicated and delicate chronic affections, they are not sufficient to meet the wants of the case. These must have special consideration and treatment by a competent physician and surgeon, the medicines and other remedial means required being selected and prepared with reference to each particular case. In order ...



Diagnosis: Top retrieved chunk is on page p0373 (Shower Bath / Sea Bathing sub-modality).
Reason: The query 'therapeutic purposes and physiological effects of hydrotherapy' caused the dense encoder
to match specific violent-reaction bath descriptions on p0373 ('shock to the nervous system') rather than
the general introductory definitions on p0364-p0366. In Stage 5 (A3), hybrid retrieval + BGE reranking resolves this.


In [8]:
# ─── 4. Out-of-Corpus Negative Query & Abstention Evaluation ───────────────
neg_query = 'What is the mechanism of action of HMG-CoA reductase inhibitors (statins) in lowering serum LDL cholesterol in coronary artery disease?'
display(Markdown('### 🛑 Out-of-Corpus Negative Query & Abstention Gate'))
display(Markdown(f'**Query**: *{neg_query}*'))

hits_neg = retrieve(neg_query, k=1)
top_score = hits_neg[0]['score']
threshold = cfg.get('retrieve', {}).get('weak_threshold', 0.55)

display(Markdown(f'**Top Chunk Score**: `{top_score:.4f}` | **Calibrated Abstention Threshold** ($\tau$): `{threshold:.2f}`'))
if top_score < threshold:
    display(Markdown(f'**Decision**: 🛑 **ABSTAINED** (Confidence {top_score:.4f} < {threshold:.2f}). Correctly avoided hallucinating 1890 medical advice for modern pharmacology.'))
else:
    display(Markdown('**Decision**: Retained for answering.'))


### 🛑 Out-of-Corpus Negative Query & Abstention Gate


**Query**: *What is the mechanism of action of HMG-CoA reductase inhibitors (statins) in lowering serum LDL cholesterol in coronary artery disease?*


**Top Chunk Score**: `0.3884` | **Calibrated Abstention Threshold** ($\tau$): `0.55`


**Decision**: 🛑 **ABSTAINED** (Confidence 0.3884 < 0.55). Correctly avoided hallucinating 1890 medical advice for modern pharmacology.
